# SPY · 42-bar image training

Train a vision model on the approved dark charts: **42 completed daily candles, EMA 8, EMA 21, and volume**. The model receives image pixels only. Excel/CSV numbers are used as labels and audit metadata; they are never added to the model's image features.

**This first experiment learns next-session open-to-close direction and candle-body uncertainty from automatic outcome labels.** It can produce `CALL_BIAS`, `PUT_BIAS`, or `WAIT` candidates. It does not yet discover a reliable “enter puts when price touches EMA 8” trigger. That later stage needs intraday observations and outcome-based evaluation. The first experiment uses automatically generated future-outcome labels.

| Split, by the next candle's date | Images | Use |
|---|---:|---|
| 2020–2024 | 1,258 | Train model weights |
| 2025 | 250 | Select checkpoint, probability calibration, and a small threshold grid |
| 2026 through November 20 | 224 | Locked final assessment |
| Latest completed 42-bar window | 1 | Unlabeled next-session candidate |

The first training image is as of 2019-12-31 because its **target** is the first 2020 session. Three exact dojis have no binary-direction label; they remain in the body/ATR task. There are 1,256 train and 249 validation binary labels.

**No model has been trained in this notebook yet. All outputs below start empty.**

## How to use

1. Open this notebook in Google Colab. A GPU runtime is optional but recommended for full training.
2. Upload **SPY_42_training_bundle_2020_2026.zip** when prompted, or set a Drive path below.
3. Run the setup, preview, and short smoke-check cells.
4. Run full training. The default uses an ImageNet-pretrained ResNet18 feature extractor and small trainable heads.
5. Inspect 2025 development results and the latest image. The final 2026 assessment stays disabled until you deliberately enable its separate cell.

The first pretrained run may download standard ResNet18 weights. The smoke check uses random weights and does not download them. This notebook does not reinstall or upgrade PyTorch automatically. Existing torch/torchvision packages must be a compatible pair.

**Persistence:** Colab runtime files are temporary. You may explicitly enable Drive persistence below, or download the results archive after training.

In [ ]:
# Configuration: change these before running the remaining cells.
from pathlib import Path
from datetime import datetime, timezone

USE_GOOGLE_DRIVE = False
DRIVE_ZIP_PATH = ""  # Example: /content/drive/MyDrive/SPY_42_training_bundle_2020_2026.zip
PERSIST_RESULTS_TO_DRIVE = False
DRIVE_RESULTS_FOLDER = "/content/drive/MyDrive/SPY42_runs"

EXPERIMENT_NAME = "spy42_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EPOCHS = 15
BATCH_SIZE = 16
NUM_WORKERS = 2  # Set to 0 if a runtime has worker/process issues.
USE_PRETRAINED_WEIGHTS = True
FINE_TUNE_LAYER4 = False  # Start with frozen pretrained features; change before final-test access.
RUN_SMOKE_TEST = True
RUN_TRAINING = True

# This is only an illustrative signed-SPY return proxy, never option PnL.
PROXY_ROUND_TRIP_COST_BPS = 0.0  # 0 explicitly means frictionless.
MIN_VALIDATION_SIGNALS = 30

In [ ]:
# Check installed packages without upgrading the runtime.
import importlib.util
import sys
import json
import subprocess
import shutil
import zipfile

required = {"torch": "torch", "torchvision": "torchvision", "numpy": "numpy",
            "pandas": "pandas", "PIL": "Pillow", "matplotlib": "matplotlib"}
missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]
if missing:
    raise RuntimeError("Missing packages: " + ", ".join(missing) +
                       ". Use a standard Colab runtime or install compatible packages manually.")
import torch
import torchvision
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__, "Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CPU mode is supported. Full training is slower; the smoke check stays small.")

In [ ]:
# Optional Drive connection happens only when you explicitly enable it above.
if USE_GOOGLE_DRIVE or PERSIST_RESULTS_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if USE_GOOGLE_DRIVE:
    if not DRIVE_ZIP_PATH:
        raise ValueError("Set DRIVE_ZIP_PATH to the accompanying dataset ZIP.")
    dataset_zip = Path(DRIVE_ZIP_PATH)
else:
    from google.colab import files
    print("Choose SPY_42_training_bundle_2020_2026.zip, which includes spy_42_dataset/.")
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
    if len(zip_names) != 1:
        raise ValueError("Upload exactly one dataset ZIP.")
    dataset_zip = Path(zip_names[0]).resolve()
    del uploaded  # Avoid retaining a second large copy in Python memory.

if not dataset_zip.is_file():
    raise FileNotFoundError(dataset_zip)
extract_root = Path("/content") / (EXPERIMENT_NAME + "_input")
extract_root.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(dataset_zip) as archive:
    for member in archive.infolist():
        destination = (extract_root / member.filename).resolve()
        if not destination.is_relative_to(extract_root.resolve()):
            raise ValueError("ZIP contains an unsafe path.")
        if ((member.external_attr >> 16) & 0o170000) == 0o120000:
            raise ValueError("ZIP symbolic links are not accepted.")
    archive.extractall(extract_root)

roots = [p.parent for p in extract_root.rglob("manifest.csv")
         if (p.parent / "dataset_config.json").is_file()]
if len(roots) != 1:
    raise ValueError("Could not uniquely find the dataset manifest/config inside the ZIP.")
DATA_ROOT = roots[0]
TRAINER_PATH = DATA_ROOT / "code" / "train_spy_images.py"
if not TRAINER_PATH.is_file():
    raise FileNotFoundError("Expected bundled code/train_spy_images.py inside the dataset ZIP.")

results_parent = (Path(DRIVE_RESULTS_FOLDER) if PERSIST_RESULTS_TO_DRIVE
                  else Path("/content/spy42_runs"))
RUN_DIR = results_parent / EXPERIMENT_NAME
results_parent.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATA_ROOT)
print("Experiment outputs:", RUN_DIR)

## Verify the image input

The trainer reads date/path metadata first. During training it skips test rows before parsing target-value columns. The test-label loader also has an explicit lock.

The native image is 1920 × 1080. The model transform crops the price panel and volume panel separately, stacks them, and resizes the pair to **768 × 336**. This preserves all 42 candles while excluding most page headings and axes. There are **no random flips, rotations, color changes, or random crops**. The pixel normalization uses fixed ImageNet constants, not a fit to the full market history.

In [ ]:
# Import the reusable trainer directly from the ZIP.
spec = importlib.util.spec_from_file_location("spy_image_trainer", TRAINER_PATH)
trainer = importlib.util.module_from_spec(spec)
spec.loader.exec_module(trainer)
DATASET_CONFIG = trainer.load_dataset_config(DATA_ROOT)
manifest_index = trainer.load_manifest_index(DATA_ROOT)  # No outcome-value columns.
display(manifest_index.groupby("split").size().rename("images").to_frame())
print("Features: image pixels only. Test outcome values remain locked.")

preview_row = manifest_index.loc[manifest_index["split"].eq("train")].iloc[-1]
with Image.open(trainer.safe_image_path(DATA_ROOT, preview_row["image_path"])) as source:
    original = source.convert("RGB")
    model_view = trainer.make_model_view(original, DATASET_CONFIG, height=336, width=768)
fig, axes = plt.subplots(2, 1, figsize=(16, 12))
axes[0].imshow(original)
axes[0].set_title("Original 42-bar chart — training sample")
axes[1].imshow(model_view)
axes[1].set_title("Exact deterministic model view: price panel + volume panel")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Short wiring check — run before full training

This forces CPU, random weights, four training images, four validation images, one epoch, and a reduced image size. It checks the forward/backward pass, checkpoint saving/reloading, calibration plumbing, and live-image inference. It never downloads pretrained weights or accesses 2026 outcome labels.

Outputs are stored in a unique `smoke_runs/smoke_...` subfolder. The smoke check cannot overwrite the real experiment. Its predictions and scores have **no predictive meaning**, and its recommendation is forced to `WAIT`.

In [ ]:
if RUN_SMOKE_TEST:
    smoke_command = [sys.executable, str(TRAINER_PATH),
                     "--data-root", str(DATA_ROOT),
                     "--output-dir", str(RUN_DIR),
                     "--smoke-test", "--workers", "0"]
    subprocess.run(smoke_command, check=True)
else:
    print("Smoke check skipped by configuration.")

## Train with 2020–2024 images; select with 2025 only

The default model has a pretrained ResNet18 image backbone, a binary-direction head, and an ordered quantile head for the next candle body divided by the previous day's Wilder ATR14. The three outputs are the 10th, 50th, and 90th percentiles; they are ordered by construction and the targets are not clipped.

The direction loss ignores exact dojis. The body/ATR loss retains them. Image backbone features are frozen by default; `FINE_TUNE_LAYER4=True` permits the last convolutional block to adapt. No metadata is fed into either head.

Checkpoint selection, a single probability-calibration temperature, and the candidate thresholds **0.55, 0.60, 0.65** all use 2025. Threshold selection requires at least 30 validation signals and a positive illustrative signed-SPY open-to-close return. If no threshold qualifies, all candidates remain `WAIT`. This development reuse means the validation results are optimistic for assessment purposes; the 2026 test remains untouched.

The proxy assumes direction at the next official open and exit at its close. It is **not option profit**, and it has no EMA-touch entry rule. A zero cost setting is explicitly frictionless.

In [ ]:
if RUN_TRAINING:
    training_command = [sys.executable, str(TRAINER_PATH),
                        "--data-root", str(DATA_ROOT),
                        "--output-dir", str(RUN_DIR),
                        "--epochs", str(EPOCHS),
                        "--batch-size", str(BATCH_SIZE),
                        "--workers", str(NUM_WORKERS),
                        "--min-validation-signals", str(MIN_VALIDATION_SIGNALS),
                        "--proxy-cost-bps", str(PROXY_ROUND_TRIP_COST_BPS),
                        "--pretrained" if USE_PRETRAINED_WEIGHTS else "--no-pretrained"]
    if FINE_TUNE_LAYER4:
        training_command.append("--fine-tune-layer4")
    subprocess.run(training_command, check=True)
else:
    print("Training skipped. Set RUN_DIR to an existing completed experiment to inspect it.")

In [ ]:
# Training/validation inspection only — this cell has no final-test access.
history_path = RUN_DIR / "training_history.csv"
if history_path.is_file():
    history = pd.read_csv(history_path)
    display(history)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(history["epoch"], history["training_loss"], label="training objective")
    axes[0].plot(history["epoch"], history["validation_objective"], label="2025 validation objective")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[1].plot(history["epoch"], history["validation_log_loss"], label="2025 direction log loss")
    axes[1].plot(history["epoch"], history["validation_pinball"], label="2025 body pinball loss")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    print("Development results only; no test conclusions:")
    print((RUN_DIR / "validation_metrics.json").read_text())
    print("Frozen calibration/threshold:")
    print((RUN_DIR / "calibration.json").read_text())
else:
    print("No completed training run at", RUN_DIR)

## Latest completed image: a direction candidate with uncertainty

This inference uses the latest 42-bar image. The model's calibrated probability concerns **next-session close versus next-session open**. It is not the probability of an option contract being profitable. The body interval is nominal; its actual coverage must be measured.

The reported EMA and ATR values are known-as-of context for your review. They do not enter the neural network. No conditional entry or option contract is selected by this first-stage daily model.

In [ ]:
latest_path = RUN_DIR / "latest_prediction.json"
if latest_path.is_file():
    latest = json.loads(latest_path.read_text())
    display(pd.DataFrame([{
        "as_of": latest["asof_date"],
        "candidate": latest["direction_candidate"],
        "model_P_next_green": latest["model_probability_next_close_above_next_open"],
        "q10_body_ATR": latest["body_over_asof_ATR14_quantiles"]["q10"],
        "median_body_ATR": latest["body_over_asof_ATR14_quantiles"]["q50"],
        "q90_body_ATR": latest["body_over_asof_ATR14_quantiles"]["q90"],
        "interval_crosses_zero": latest["nominal_interval_crosses_zero"],
    }]))
    print(latest["entry_trigger"])
    print(latest["probability_notice"])
else:
    print("Train the model first. No result has been fabricated.")

## Locked final assessment: 2026 through November 20

**Leave the switch below False during development.** Once the architecture, settings, calibration, and thresholds are fixed, change it to True and run this cell explicitly. A normal **Run all** with the delivered defaults will skip it.

This is the only notebook cell that loads 2026 outcome labels. It compares the frozen model with the training-period majority/frequency direction baseline and a zero-body-return baseline, and reports the optional signed-SPY proxy under the frozen threshold.

After you inspect this report, tuning based on it makes 2026 development data. A later model then needs genuinely new future observations for another untouched assessment. The evaluator refuses to overwrite an existing final-test output folder.

In [ ]:
RUN_FINAL_TEST = False  # Deliberate, explicit opt-in; delivered disabled.

if RUN_FINAL_TEST:
    if not (RUN_DIR / "best_model.pt").is_file():
        raise FileNotFoundError("Complete full training before final assessment.")
    final_command = [sys.executable, str(TRAINER_PATH),
                     "--data-root", str(DATA_ROOT),
                     "--run-dir", str(RUN_DIR),
                     "--final-test", "--batch-size", str(BATCH_SIZE),
                     "--workers", str(NUM_WORKERS)]
    subprocess.run(final_command, check=True)
    test_metrics = json.loads((RUN_DIR / "final_test" / "test_metrics.json").read_text())
    print(json.dumps(test_metrics, indent=2))
else:
    print("Final test stays LOCKED. No 2026 outcome labels loaded or evaluated.")

## Save the completed run

A completed experiment contains the checkpoint, training history, validation predictions, calibration/threshold settings, configuration, and latest-image candidate. A final-test report appears only if you deliberately enabled the preceding cell. The following archive excludes smoke runs to keep the download focused on the real experiment.

If Drive persistence was enabled, those files are already written under the chosen Drive folder. The download below is optional and user initiated.

In [ ]:
DOWNLOAD_RESULTS = False

if DOWNLOAD_RESULTS:
    if not (RUN_DIR / "run_config.json").is_file():
        raise FileNotFoundError("No completed full experiment to archive.")
    results_zip = Path("/content") / (EXPERIMENT_NAME + "_results.zip")
    with zipfile.ZipFile(results_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in RUN_DIR.rglob("*"):
            if path.is_file() and "smoke_runs" not in path.relative_to(RUN_DIR).parts:
                archive.write(path, Path(EXPERIMENT_NAME) / path.relative_to(RUN_DIR))
    from google.colab import files
    files.download(str(results_zip))
else:
    print("Download disabled. Outputs are at", RUN_DIR)

## What this experiment can establish

- Whether your **42-bar chart images** carry useful next-session directional/body information under this chronological evaluation.
- Whether the vision model beats simple benchmarks on the reserved period.
- How often it waits, how its nominal uncertainty interval behaves, and whether its probabilities are informative.

It does not establish an executable options strategy, a particular strike/expiry, or the order of next-day intraday price events. Your intended conditional EMA recommendations are a later stage: collect intraday visual observations available at each possible entry and evaluate candidate actions against later outcomes, including waiting and invalidating a setup. That can also use automatically generated outcome labels.

### Reusable command-line equivalents

```bash
python train_spy_images.py --data-root /path/to/spy_42_dataset --output-dir /path/to/run --smoke-test
python train_spy_images.py --data-root /path/to/spy_42_dataset --output-dir /path/to/run --epochs 15 --pretrained
python train_spy_images.py --data-root /path/to/spy_42_dataset --run-dir /path/to/run --predict-live
# Explicit final assessment, only after settings are fixed:
python train_spy_images.py --data-root /path/to/spy_42_dataset --run-dir /path/to/run --final-test
```

### References for the design

- Jiang, Kelly, and Xiu, *[“(Re-)Imag(in)ing Price Trends”](https://economics.yale.edu/research/re-imagining-price-trends)*, Journal of Finance (2023): precedent for image-based return research; no guarantee for this SPY task.
- TradingView, *[Repainting](https://www.tradingview.com/pine-script-docs/concepts/repainting/)*: historical daily candles do not retain the intraday event sequence, and live OHLC/indicator values can change.
- scikit-learn, *[TimeSeriesSplit](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html)* and *[data leakage](https://scikit-learn.org/stable/common_pitfalls.html#data-leakage)*: chronological evaluation and training-only fitting.
- The Options Industry Council, *[Option Price Behavior](https://www.optionseducation.org/referencelibrary/faq/option-price-behavior)*: option returns also depend on strike, expiry, volatility, and other pricing inputs.